In [2]:
from pathlib import Path
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [3]:
PROJECT_DIR = Path("../")

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_PATH = RAW_DATA_DIR / "train.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_DIR.exists())
print("Train exists:", TRAIN_PATH.exists())
print("Processed dir exists:", PROCESSED_DATA_DIR.exists())

Project exists: True
Train exists: True
Processed dir exists: True


In [4]:
train_df = pd.read_csv(TRAIN_PATH)

print("Train shape:", train_df.shape)
display(train_df.head())
print("Columns:", train_df.columns.tolist())

Train shape: (9500, 3)


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


Columns: ['id', 'prompt', 'answer']


In [5]:
def find_column(df, possible_names):
    columns_lower = {col.lower(): col for col in df.columns}
    
    for name in possible_names:
        if name.lower() in columns_lower:
            return columns_lower[name.lower()]
    
    return None


prompt_col = find_column(
    train_df,
    ["prompt", "question", "puzzle", "input", "problem"]
)

answer_col = find_column(
    train_df,
    ["answer", "target", "output", "solution", "label"]
)

print("Prompt column:", prompt_col)
print("Answer column:", answer_col)

if prompt_col is None:
    raise ValueError(f"Could not find prompt column. Columns: {train_df.columns.tolist()}")

if answer_col is None:
    raise ValueError(f"Could not find answer column. Columns: {train_df.columns.tolist()}")

Prompt column: prompt
Answer column: answer


In [6]:
real_df = train_df[[prompt_col, answer_col]].copy()
real_df.columns = ["prompt", "answer"]

real_df["prompt"] = real_df["prompt"].astype(str).str.strip()
real_df["answer"] = real_df["answer"].astype(str).str.strip()

real_df = real_df.dropna()
real_df = real_df.drop_duplicates()

real_df["source"] = "official"

print("Clean official shape:", real_df.shape)
display(real_df.head())

Clean official shape: (9500, 3)


,prompt,answer,source
0,"In Alice's Wonderland, a secret bit manipulati...",10010111,official
1,"In Alice's Wonderland, a secret bit manipulati...",01000011,official
2,"In Alice's Wonderland, secret encryption rules...",cat imagines book,official
3,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,official
4,"In Alice's Wonderland, secret encryption rules...",wizard creates secret,official


In [7]:
pd.set_option("display.max_colwidth", 1000)

display(real_df.sample(min(10, len(real_df)), random_state=42))

,prompt,answer,source
952,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01001000 -> 10100100\n00011100 -> 11001110\n01001001 -> 10110100\n00011111 -> 11111111\n01000100 -> 01100010\n01010110 -> 01001011\n11000101 -> 00110010\n10010101 -> 00011010\n00000100 -> 01000010\n\nNow, determine the output for: 00101111",11100111,official
6599,"In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n<[+&< = <[&<\n)&*#) = ){}}\n}{+{\ = }{{\\n{\*#} = {@#{\n{&*^< = )[<<\nNow, determine the result for: \&+[[",\&[[,official
4220,"In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:\n99 -> XCIX\n15 -> XV\n73 -> LXXIII\nNow, write the number 25 in the Wonderland numeral system.",XXV,official
2554,"In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:\n26.92 m becomes 37.82\n38.94 m becomes 54.71\n34.29 m becomes 48.18\nNow, convert the following measurement: 14.75 m",20.72,official
4519,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n10100110 -> 01111010\n11101010 -> 01001111\n01111110 -> 00100000\n11011010 -> 01011011\n11001111 -> 11010100\n00010101 -> 10001111\n11000010 -> 01010001\n10100000 -> 01111000\n11001001 -> 11010110\n00010011 -> 10001101\n\nNow, determine the output for: 00110111",10010110,official
1557,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01001011 -> 01100111\n00100000 -> 01111101\n11111011 -> 10101111\n00010010 -> 10010110\n11000101 -> 10110111\n10101101 -> 10010011\n00000100 -> 10101111\n\nNow, determine the output for: 11011001",00000111,official
7945,"In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:\n&&]{> = "":]\n>&*%| = &/{{\n$!]$& = /$\nNow, determine the result for: %&#$>","{/""",official
4667,"In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:\n91 -> XCI\n94 -> XCIV\n1 -> I\n95 -> XCV\nNow, write the number 56 in the Wonderland numeral system.",LVI,official
7697,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01111000 -> 11011100\n10111100 -> 10101110\n10101000 -> 11110100\n10111000 -> 10111100\n00000011 -> 00001101\n11011111 -> 00010011\n01000110 -> 00111011\n11100111 -> 11101111\n10011011 -> 00100001\n11011100 -> 00011110\n\nNow, determine the output for: 10000000",01000000,official
8928,"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nlqgnbqj qafecjqo gschq hveegtq -> teacher explores above village\npvwgjm oqqo rymqj hveegtq -> wizard sees under village\nolrmqyl njqglqo lbq oljgytq frwweq -> student creates the strange puzzle\nxvyt pglnbqo rymqj pcymqjegym -> king watches under wonderland\nlqgnbqj njqglqo lbq oqnjql ljqgorjq -> teacher creates the secret treasure\nNow, decrypt the following text: pvwgjm zceecpo vyovmq pcymqjegym",wizard follows inside wonderland,official


### Build symbol set from official data

In [8]:
official_text = " ".join(real_df["prompt"].astype(str).tolist())
official_answers = " ".join(real_df["answer"].astype(str).tolist())

all_chars = sorted(set(official_text + official_answers))

bad_chars = set([" ", "\n", "\t"])
SYMBOLS = [char for char in all_chars if char not in bad_chars]

if len(SYMBOLS) < 10:
    SYMBOLS = list("$?<>`\\{@):;#%&*![]{}+-=/|~^_")

print("Number of symbols:", len(SYMBOLS))
print(SYMBOLS[:100])

Number of symbols: 81
['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'F', 'H', 'I', 'L', 'N', 'O', 'R', 'S', 'T', 'V', 'W', 'X', '[', '\\', ']', '^', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}']


### Random Seeds

In [9]:
random.seed(42)
np.random.seed(42)

### Rule Cless

In [10]:
class PuzzleRule:
    def __init__(self, name, transform_func):
        self.name = name
        self.transform_func = transform_func

    def apply(self, text):
        return self.transform_func(text)

### Helper function

In [11]:
def safe_apply(text, func):
    try:
        output = func(text)
        if output is None:
            return text
        return str(output)
    except Exception:
        return text

### Core Rules

In [12]:
def remove_first_char(text):
    return text[1:] if len(text) > 1 else text


def remove_last_char(text):
    return text[:-1] if len(text) > 1 else text


def duplicate_first_char(text):
    return text[0] + text if len(text) > 0 else text


def duplicate_last_char(text):
    return text + text[-1] if len(text) > 0 else text


def swap_first_two(text):
    if len(text) < 2:
        return text
    return text[1] + text[0] + text[2:]


def swap_last_two(text):
    if len(text) < 2:
        return text
    return text[:-2] + text[-1] + text[-2]


def reverse_text(text):
    return text[::-1]


def remove_middle_char(text):
    if len(text) < 3:
        return text
    mid = len(text) // 2
    return text[:mid] + text[mid + 1:]


def duplicate_middle_char(text):
    if len(text) < 3:
        return text
    mid = len(text) // 2
    return text[:mid] + text[mid] + text[mid:]


def rotate_left(text):
    if len(text) < 2:
        return text
    return text[1:] + text[0]


def rotate_right(text):
    if len(text) < 2:
        return text
    return text[-1] + text[:-1]


def keep_even_positions(text):
    return text[::2] if len(text) > 1 else text


def keep_odd_positions(text):
    return text[1::2] if len(text) > 2 else text


simple_rules = [
    PuzzleRule("remove_first_char", remove_first_char),
    PuzzleRule("remove_last_char", remove_last_char),
    PuzzleRule("duplicate_first_char", duplicate_first_char),
    PuzzleRule("duplicate_last_char", duplicate_last_char),
    PuzzleRule("swap_first_two", swap_first_two),
    PuzzleRule("swap_last_two", swap_last_two),
    PuzzleRule("reverse_text", reverse_text),
    PuzzleRule("remove_middle_char", remove_middle_char),
    PuzzleRule("duplicate_middle_char", duplicate_middle_char),
    PuzzleRule("rotate_left", rotate_left),
    PuzzleRule("rotate_right", rotate_right),
    PuzzleRule("keep_even_positions", keep_even_positions),
    PuzzleRule("keep_odd_positions", keep_odd_positions),
]

print("Simple rules:", len(simple_rules))

Simple rules: 13


### Replacement Rules

In [13]:
def replace_symbol_factory(old, new):
    def replace_symbol(text):
        return text.replace(old, new)
    return replace_symbol


replacement_rules = []

sample_symbols = SYMBOLS[:40]

for old in sample_symbols:
    for new in sample_symbols:
        if old != new:
            replacement_rules.append(
                PuzzleRule(
                    f"replace_{old}_with_{new}",
                    replace_symbol_factory(old, new)
                )
            )

print("Replacement rules:", len(replacement_rules))

Replacement rules: 1560


### Position replacement rules

In [14]:
def replace_first_char_factory(new):
    def replace_first_char(text):
        if len(text) == 0:
            return text
        return new + text[1:]
    return replace_first_char


def replace_last_char_factory(new):
    def replace_last_char(text):
        if len(text) == 0:
            return text
        return text[:-1] + new
    return replace_last_char


def replace_middle_char_factory(new):
    def replace_middle_char(text):
        if len(text) < 3:
            return text
        mid = len(text) // 2
        return text[:mid] + new + text[mid + 1:]
    return replace_middle_char


position_rules = []

for symbol in sample_symbols:
    position_rules.append(PuzzleRule(f"replace_first_with_{symbol}", replace_first_char_factory(symbol)))
    position_rules.append(PuzzleRule(f"replace_last_with_{symbol}", replace_last_char_factory(symbol)))
    position_rules.append(PuzzleRule(f"replace_middle_with_{symbol}", replace_middle_char_factory(symbol)))

print("Position rules:", len(position_rules))

Position rules: 120


### Two-step composie rules

In [15]:
def compose_rules(rule_a, rule_b):
    def composed(text):
        return rule_b.apply(rule_a.apply(text))
    return composed


composite_rules = []

base_for_composition = simple_rules[:]

for rule_a in base_for_composition:
    for rule_b in base_for_composition:
        if rule_a.name != rule_b.name:
            composite_rules.append(
                PuzzleRule(
                    f"{rule_a.name}_then_{rule_b.name}",
                    compose_rules(rule_a, rule_b)
                )
            )

print("Composite rules:", len(composite_rules))

Composite rules: 156


### Combine rules with better weighting

In [16]:
rules = []

rules.extend(simple_rules * 8)
rules.extend(position_rules * 3)
rules.extend(replacement_rules * 1)
rules.extend(composite_rules * 2)

print("Total weighted rules:", len(rules))

Total weighted rules: 2336


### Random String Generator

In [17]:
def random_string(min_len=4, max_len=9):
    length = random.randint(min_len, max_len)
    return "".join(random.choice(SYMBOLS) for _ in range(length))

### Build Puzzle

In [18]:
def make_puzzle(num_examples=None):
    if num_examples is None:
        num_examples = random.choice([3, 4, 4, 5])

    rule = random.choice(rules)

    examples = []
    used_inputs = set()

    attempts = 0

    while len(examples) < num_examples and attempts < 200:
        attempts += 1

        x = random_string()
        y = rule.apply(x)

        if x in used_inputs:
            continue

        if y == x:
            continue

        if len(y) == 0:
            continue

        used_inputs.add(x)
        examples.append((x, y))

    if len(examples) < num_examples:
        return None

    target = random_string()
    answer = rule.apply(target)

    if answer == target:
        return None

    if len(answer) == 0:
        return None

    prompt_lines = []

    intro_options = [
        "In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:",
        "A hidden transformation rule is applied to the following examples:",
        "Find the secret transformation rule from these examples:",
        "A symbolic rule changes each input into an output:"
    ]

    prompt_lines.append(random.choice(intro_options))

    for x, y in examples:
        prompt_lines.append(f"{x} = {y}")

    target_lines = [
        f"Now, determine the result for: {target}",
        f"What is the result for: {target}",
        f"Apply the same rule to: {target}"
    ]

    prompt = "\n".join(prompt_lines)
    prompt += "\n" + random.choice(target_lines)

    return {
        "prompt": prompt,
        "answer": answer,
        "rule": rule.name,
        "source": "synthetic_v2"
    }

### Generate stronger synthetic data

In [19]:
NUM_SYNTHETIC = 8000

synthetic_rows = []

while len(synthetic_rows) < NUM_SYNTHETIC:
    item = make_puzzle()
    
    if item is not None:
        synthetic_rows.append(item)

synthetic_df = pd.DataFrame(synthetic_rows)

print("Synthetic shape:", synthetic_df.shape)
display(synthetic_df.head())

Synthetic shape: (8000, 4)


,prompt,answer,rule,source
0,"Find the secret transformation rule from these examples:\nFA>2.r,xc = F>.,c\n$,=? = $=\nz$t:rb>f = ztr>\nNow, determine the result for: F!5cTF4=",F5T4,keep_even_positions,synthetic_v2
1,"A symbolic rule changes each input into an output:\n""Cq0g[C = >Cq0g[C\n\[.?i$|t = >[.?i$|t\n{>)hLa = >>)hLa\n2&%L = >&%L\n/-@q2]g = >-@q2]g\nApply the same rule to: rbx4b-",>bx4b-,replace_first_with_>,synthetic_v2
2,"Find the secret transformation rule from these examples:\n2BIR0 = 4BIR0\nk9>24* = k9>44*\nb5:z2B'j = b5:z4B'j\n%Hi2i = %Hi4i\n}.wS2&WrT = }.wS4&WrT\nNow, determine the result for: 7hj}82)g%",7hj}84)g%,replace_2_with_4,synthetic_v2
3,"A hidden transformation rule is applied to the following examples:\n0o4L = 04\n5R>Vo = 5>o\n+B:sF1 = +:F\nL{q,m6xw4 = Lqmx4\n|zTu& = |T&\nApply the same rule to: +&vC",+v,duplicate_first_char_then_keep_odd_positions,synthetic_v2
4,"Find the secret transformation rule from these examples:\n}rILjA` = }rI'jA`\ng*(5eb = g*('eb\nh<Tz3OO = h<T'3OO\nV`1[nt.O@ = V`1['t.O@\n0DfA3-' = 0Df'3-'\nNow, determine the result for: ]{bA5R",]{b'5R,replace_middle_with_',synthetic_v2


### Quality Check

In [20]:
print("Missing values:")
display(synthetic_df.isna().sum())

duplicate_count = synthetic_df.duplicated(subset=["prompt", "answer"]).sum()
print("Duplicate prompt-answer rows:", duplicate_count)

rule_counts = (
    synthetic_df["rule"]
    .value_counts()
    .reset_index()
)

rule_counts.columns = ["rule", "count"]

display(rule_counts.head(30))

Missing values:


prompt    0
answer    0
rule      0
source    0
dtype: int64

Duplicate prompt-answer rows: 0


,rule,count
0,duplicate_first_char,91
1,duplicate_middle_char,83
2,rotate_right,81
3,remove_last_char,81
4,keep_odd_positions,81
5,swap_first_two,78
6,duplicate_last_char,74
7,remove_middle_char,70
8,keep_even_positions,70
9,remove_first_char,67


### Preview synthetic examples

In [21]:
for i, row in synthetic_df.sample(10, random_state=42).iterrows():
    print("PROMPT:")
    print(row["prompt"])
    print("ANSWER:", row["answer"])
    print("RULE:", row["rule"])
    print("-" * 100)

PROMPT:
A hidden transformation rule is applied to the following examples:
OHm+f5 = Hm+5fO
-9le, = 9l,e-
&-C%[^5y = -C%[^y5&
y/-xx|xs = /-xx|sxy
Apply the same rule to: {wW>
ANSWER: w>W{
RULE: swap_last_two_then_rotate_left
----------------------------------------------------------------------------------------------------
PROMPT:
A hidden transformation rule is applied to the following examples:
/0b.B = /0b.BB
lXI"Dp34o = lXI"Dp34oo
gd(Ryso-$ = gd(Ryso-$$
Apply the same rule to: kj\-ml
ANSWER: kj\-mll
RULE: duplicate_last_char_then_swap_last_two
----------------------------------------------------------------------------------------------------
PROMPT:
A symbolic rule changes each input into an output:
]^eHqHXi= = ]^eH-HXi=
.}:#3 = .}-#3
`%%W97 = `%%-97
$suf = $s-f
Apply the same rule to: eRy'C\"L
ANSWER: eRy'-\"L
RULE: replace_middle_with_-
----------------------------------------------------------------------------------------------------
PROMPT:
A hidden transformation rule is appl

### Combine official and synthetic

In [22]:
official_boost_df = pd.concat(
    [real_df[["prompt", "answer", "source"]]] * 3,
    ignore_index=True
)

combined_df = pd.concat(
    [
        official_boost_df,
        synthetic_df[["prompt", "answer", "source"]]
    ],
    ignore_index=True
)

combined_df = combined_df.drop_duplicates(subset=["prompt", "answer"]).reset_index(drop=True)

print("Combined shape:", combined_df.shape)
display(combined_df["source"].value_counts())
display(combined_df.head())

Combined shape: (17500, 3)


source
official        9500
synthetic_v2    8000
Name: count, dtype: int64

,prompt,answer,source
0,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100",10010111,official
1,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n10001110 -> 00100110\n10011001 -> 01000100\n01100100 -> 00010001\n10000010 -> 00001010\n00011011 -> 01001100\n00111010 -> 10011100\n01101111 -> 00110111\n10010110 -> 01011010\n00001010 -> 00101100\n\nNow, determine the output for: 11100000",01000011,official
2,"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley\npqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle\ngbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door\nbxo sfjpov pqrsfv dfjjfig -> the golden dragon follows\nnqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious secret\nNow, decrypt the following text: trb wzrswvog hffk",cat imagines book,official
3,"In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:\n11 -> XI\n15 -> XV\n94 -> XCIV\n19 -> XIX\nNow, write the number 38 in the Wonderland numeral system.",XXXVIII,official
4,"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nwkgqa lsrqaq wneeke -> mouse chases mirror\nwkgqa nwrjnvaq nv brmrla -> mouse imagines in palace\nsrppae oerhq gvoae wkgvprnv -> hatter draws under mountain\npsa qaleap hncreo onqlkzaeq -> the secret wizard discovers\npsa hnqa xneo earoq -> the wise bird reads\nNow, decrypt the following text: hncreo learpaq qaleap",wizard creates secret,official


### Format SFT text

In [23]:
SYSTEM_PROMPT = """You are solving a reasoning puzzle.
Return only the final answer."""


def format_sft_text(prompt, answer):
    return f"""{SYSTEM_PROMPT}

Puzzle:
{prompt}

Answer:
{answer}"""


combined_df["text"] = combined_df.apply(
    lambda row: format_sft_text(row["prompt"], row["answer"]),
    axis=1
)

display(combined_df[["text", "source"]].head())

,text,source
0,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100\n\nAnswer:\n10010111",official
1,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n10001110 -> 00100110\n10011001 -> 01000100\n01100100 -> 00010001\n10000010 -> 00001010\n00011011 -> 01001100\n00111010 -> 10011100\n01101111 -> 00110111\n10010110 -> 01011010\n00001010 -> 00101100\n\nNow, determine the output for: 11100000\n\nAnswer:\n01000011",official
2,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley\npqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle\ngbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door\nbxo sfjpov pqrsfv dfjjfig -> the golden dragon follows\nnqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious secret\nNow, decrypt the following text: trb wzrswvog hffk\n\nAnswer:\ncat imagines book",official
3,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:\n11 -> XI\n15 -> XV\n94 -> XCIV\n19 -> XIX\nNow, write the number 38 in the Wonderland numeral system.\n\nAnswer:\nXXXVIII",official
4,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nwkgqa lsrqaq wneeke -> mouse chases mirror\nwkgqa nwrjnvaq nv brmrla -> mouse imagines in palace\nsrppae oerhq gvoae wkgvprnv -> hatter draws under mountain\npsa qaleap hncreo onqlkzaeq -> the secret wizard discovers\npsa hnqa xneo earoq -> the wise bird reads\nNow, decrypt the following text: hncreo learpaq qaleap\n\nAnswer:\nwizard creates secret",official


### Lenth Check

In [24]:
combined_df["prompt_length"] = combined_df["prompt"].str.len()
combined_df["answer_length"] = combined_df["answer"].str.len()
combined_df["text_length"] = combined_df["text"].str.len()

display(
    combined_df[
        ["prompt_length", "answer_length", "text_length"]
    ].describe()
)

,prompt_length,answer_length,text_length
count,17500.00000,17500.000000,17500.000000
mean,241.71760,7.360514,334.078114
std,102.45678,6.182920,105.351561
min,114.00000,1.000000,200.000000
25%,168.00000,4.000000,259.000000
50%,206.00000,5.000000,295.000000
75%,285.00000,8.000000,376.000000
max,510.00000,39.000000,603.000000


### Remove extreme long examples

In [25]:
max_length = combined_df["text_length"].quantile(0.99)

filtered_df = combined_df[combined_df["text_length"] <= max_length].copy()
filtered_df = filtered_df.reset_index(drop=True)

print("Before filtering:", combined_df.shape)
print("After filtering:", filtered_df.shape)
print("Max kept text length:", filtered_df["text_length"].max())

Before filtering: (17500, 7)
After filtering: (17500, 7)
Max kept text length: 603


### Split train and valid

In [26]:
train_sft_df, val_sft_df = train_test_split(
    filtered_df,
    test_size=0.05,
    random_state=42,
    shuffle=True,
    stratify=filtered_df["source"]
)

print("Train shape:", train_sft_df.shape)
print("Validation shape:", val_sft_df.shape)

print("\nTrain source distribution:")
display(train_sft_df["source"].value_counts())

print("\nValidation source distribution:")
display(val_sft_df["source"].value_counts())

Train shape: (16625, 7)
Validation shape: (875, 7)

Train source distribution:


source
official        9025
synthetic_v2    7600
Name: count, dtype: int64


Validation source distribution:


source
official        475
synthetic_v2    400
Name: count, dtype: int64

### Save V2 files

In [27]:
train_csv_path = PROCESSED_DATA_DIR / "improved_train_sft_v2.csv"
val_csv_path = PROCESSED_DATA_DIR / "improved_val_sft_v2.csv"
full_csv_path = PROCESSED_DATA_DIR / "improved_full_sft_v2.csv"

train_sft_df.to_csv(train_csv_path, index=False)
val_sft_df.to_csv(val_csv_path, index=False)
filtered_df.to_csv(full_csv_path, index=False)

print("Saved CSV files:")
print(train_csv_path)
print(val_csv_path)
print(full_csv_path)

Saved CSV files:
..\data\processed\improved_train_sft_v2.csv
..\data\processed\improved_val_sft_v2.csv
..\data\processed\improved_full_sft_v2.csv


### Save V2 JSONL

In [28]:
train_jsonl_path = PROCESSED_DATA_DIR / "improved_train_sft_v2.jsonl"
val_jsonl_path = PROCESSED_DATA_DIR / "improved_val_sft_v2.jsonl"
full_jsonl_path = PROCESSED_DATA_DIR / "improved_full_sft_v2.jsonl"

train_sft_df[["text"]].to_json(
    train_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

val_sft_df[["text"]].to_json(
    val_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

filtered_df[["text"]].to_json(
    full_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved JSONL files:")
print(train_jsonl_path)
print(val_jsonl_path)
print(full_jsonl_path)

Saved JSONL files:
..\data\processed\improved_train_sft_v2.jsonl
..\data\processed\improved_val_sft_v2.jsonl
..\data\processed\improved_full_sft_v2.jsonl


### Final check

In [29]:
print("Processed files:")

for file in sorted(PROCESSED_DATA_DIR.iterdir()):
    print(file.name, round(file.stat().st_size / 1024, 2), "KB")

Processed files:
full_sft.csv 6710.46 KB
full_sft.jsonl 3902.32 KB
improved_full_sft.csv 8410.22 KB
improved_full_sft.jsonl 4984.42 KB
improved_full_sft_v2.csv 10478.93 KB
improved_full_sft_v2.jsonl 6154.12 KB
improved_train_sft.csv 7983.69 KB
improved_train_sft.jsonl 4732.05 KB
improved_train_sft_v2.csv 9948.56 KB
improved_train_sft_v2.jsonl 5843.0 KB
improved_val_sft.csv 426.59 KB
improved_val_sft.jsonl 252.37 KB
improved_val_sft_v2.csv 530.44 KB
improved_val_sft_v2.jsonl 311.11 KB
sft_train.jsonl 6983.7 KB
sft_val.jsonl 1238.41 KB
test_prompts.jsonl 3.56 KB
train_sft.csv 6368.81 KB
train_sft.jsonl 3704.01 KB
val_sft.csv 341.71 KB
val_sft.jsonl 198.3 KB


### Preview final training example

In [30]:
preview_df = pd.read_json(full_jsonl_path, lines=True)

print("Preview shape:", preview_df.shape)
display(preview_df.head())

print("\nFirst training example:")
print(preview_df.loc[0, "text"])

Preview shape: (17500, 1)


,text
0,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100\n\nAnswer:\n10010111"
1,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n10001110 -> 00100110\n10011001 -> 01000100\n01100100 -> 00010001\n10000010 -> 00001010\n00011011 -> 01001100\n00111010 -> 10011100\n01101111 -> 00110111\n10010110 -> 01011010\n00001010 -> 00101100\n\nNow, determine the output for: 11100000\n\nAnswer:\n01000011"
2,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley\npqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle\ngbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door\nbxo sfjpov pqrsfv dfjjfig -> the golden dragon follows\nnqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious secret\nNow, decrypt the following text: trb wzrswvog hffk\n\nAnswer:\ncat imagines book"
3,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:\n11 -> XI\n15 -> XV\n94 -> XCIV\n19 -> XIX\nNow, write the number 38 in the Wonderland numeral system.\n\nAnswer:\nXXXVIII"
4,"You are solving a reasoning puzzle.\nReturn only the final answer.\n\nPuzzle:\nIn Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nwkgqa lsrqaq wneeke -> mouse chases mirror\nwkgqa nwrjnvaq nv brmrla -> mouse imagines in palace\nsrppae oerhq gvoae wkgvprnv -> hatter draws under mountain\npsa qaleap hncreo onqlkzaeq -> the secret wizard discovers\npsa hnqa xneo earoq -> the wise bird reads\nNow, decrypt the following text: hncreo learpaq qaleap\n\nAnswer:\nwizard creates secret"



First training example:
You are solving a reasoning puzzle.
Return only the final answer.

Puzzle:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100

Answer:
10010111
